# Altered States — Jane Street, March 2014

[https://www.janestreet.com/puzzles/altered-states-index/](https://www.janestreet.com/puzzles/altered-states-index/)

## My Solution: 119 points
```
C M N T C
Z O R A H
L I K T D
I A S O U
D N H I L
```

| Points | State |
|---:|---|
| 13 | NORTHCAROLINA |
| 13 | SOUTHCAROLINA |
| 11 | NORTHDAKOTA |
| 11 | SOUTHDAKOTA |
| 9 | LOUISIANA |
| 8 | ARKANSAS |
| 8 | COLORADO |
| 7 | ARIZONA |
| 7 | INDIANA |
| 7 | MONTANA |
| 6 | ALASKA |
| 6 | KANSAS |
| 5 | IDAHO |
| 4 | OHIO |
| 4 | UTAH |


### Old Solution: 118 points
```
C E G F C
H A R O L
D T K I N
U O S A D
L I H N L
```

| Points | State |
|---:|---|
| 13 | NORTHCAROLINA |
| 13 | SOUTHCAROLINA |
| 11 | NORTHDAKOTA |
| 11 | SOUTHDAKOTA |
| 9 | LOUISIANA |
| 8 | ARKANSAS |
| 8 | COLORADO |
| 7 | FLORIDA |
| 7 | INDIANA |
| 6 | ALASKA |
| 6 | KANSAS |
| 6 | OREGON |
| 5 | IDAHO |
| 4 | OHIO |
| 4 | UTAH |

Found by CP-SAT with the four `NORTH`/`SOUTH` × `CAROLINA`/`DAKOTA` states required, as well as 
`ARKANSAS` and `KANSAS`.


## Goals

Fill a 5x5 grid with letters so that as many of the 50 U.S. states as possible can be spelled by
walking around the grid. This is a great hard version of an *optimization* puzzle so I can get practice with CP-SAT model.

## AI use disclaimer

I used AI to help me with Python syntax and debugging and in writing some of the more tedious helper functions.  I had it transcribe the rules as well.  It helped me understand how to implement the BFS constraints for the solver.

## Rules

- Enter letters into a 5x5 grid.
- A state is **present** if its name (spaces removed, so `RHODEISLAND`, `NEWHAMPSHIRE`) can be
  spelled by a sequence of *King's moves*: each letter after the first must sit on a square
  horizontally, vertically, or diagonally adjacent to the previous letter's square.
- A state scores its own length: `OREGON` scores 6, `RHODEISLAND` scores 11.
- A state that can be spelled more than one way still scores once.
- Maximize the total score.

## The example

The 3x3 example grid

```
. I H
D A O
. W .
```

scores **13**, from `IDAHO` (5), `IOWA` (4), and `OHIO` (4). It misses HAWAII because you can't choose not to move, so no way to double the "i"

In [1]:
from ortools.sat.python import cp_model

## The 50 states, and the grid size

In [2]:
STATE_NAMES = [
    "ALABAMA",
    "ALASKA",
    "ARIZONA",
    "ARKANSAS",
    "CALIFORNIA",
    "COLORADO",
    "CONNECTICUT",
    "DELAWARE",
    "FLORIDA",
    "GEORGIA",
    "HAWAII",
    "IDAHO",
    "ILLINOIS",
    "INDIANA",
    "IOWA",
    "KANSAS",
    "KENTUCKY",
    "LOUISIANA",
    "MAINE",
    "MARYLAND",
    "MASSACHUSETTS",
    "MICHIGAN",
    "MINNESOTA",
    "MISSISSIPPI",
    "MISSOURI",
    "MONTANA",
    "NEBRASKA",
    "NEVADA",
    "NEWHAMPSHIRE",
    "NEWJERSEY",
    "NEWMEXICO",
    "NEWYORK",
    "NORTHCAROLINA",
    "NORTHDAKOTA",
    "OHIO",
    "OKLAHOMA",
    "OREGON",
    "PENNSYLVANIA",
    "RHODEISLAND",
    "SOUTHCAROLINA",
    "SOUTHDAKOTA",
    "TENNESSEE",
    "TEXAS",
    "UTAH",
    "VERMONT",
    "VIRGINIA",
    "WASHINGTON",
    "WESTVIRGINIA",
    "WISCONSIN",
    "WYOMING",
]

GRID_SIZE = 5

ALPHABET = "ABCDEFGHIJKLMNOPRSTUVWXYZ"  # no Q needed
BLANK = "."

### Useful helper functions

Next, we need to implement a series of useful helper functions we will use.  king_neighbors is especially useful, and we must be sure not to allow the starting square into the list of neightbors!  

In [3]:
def all_cells(size):
    """Return every (row, column) pair of a square grid, reading across then down."""
    cells = []
    for row in range(size):
        for column in range(size):
            cells.append((row, column))
    return cells


def king_neighbors(row, column, size):
    """Return the cells a King can move to from (row, column).

    The eight surrounding cells, minus any that fall off the board. The starting cell itself is
    never included
    """
    neighbors = []
    for row_step in (-1, 0, 1):
        for column_step in (-1, 0, 1):
            is_staying_put = row_step == 0 and column_step == 0
            if is_staying_put:
                continue
            neighbor_row = row + row_step
            neighbor_column = column + column_step
            row_on_board = 0 <= neighbor_row < size
            column_on_board = 0 <= neighbor_column < size
            if row_on_board and column_on_board:
                neighbors.append((neighbor_row, neighbor_column))
    return neighbors

## Building the scoring function

We will now build a function that takes as input a grid of letters, and outputs a score. This will be the funtion our solver will try to maximize, although the solver won't actually see it directly -- this is just for us to be able to score our own example grids and verify the solvers solution.

Our generall strategy for this will be to, one by one, search for each state using breadth-first sweep over the word.  For speed implications, we only need to keep track of the ending square of a given search -- how we get there does not matter and there is no point in keeping track of the path, only which square we're currently on and what we have left.  

In [4]:
def word_in_grid(grid, word):
    size = len(grid)
    frontier = set()  # This handles deduping

    for row, column in all_cells(size):
        if grid[row][column] == word[0]:
            frontier.add((row, column))

    if len(frontier) == 0:
        return False

    for next_letter in word[1:]:
        new_frontier = set()  # This handles deduping
        for row, column in frontier:
            for neighbor in king_neighbors(row, column, size):
                neighbor_row, neighbor_column = neighbor
                if grid[neighbor_row][neighbor_column] == next_letter:
                    new_frontier.add(neighbor)
        frontier = new_frontier
        if len(frontier) == 0:
            return False

    return True


def states_in_grid(grid, states=STATE_NAMES):
    """Return {state: one path spelling it} for every state present in the grid."""
    found = set()
    for state in states:
        if word_in_grid(grid, state):
            found.add(state)
    return found


def score_grid(grid, states=STATE_NAMES):
    """Return the puzzle score of a grid: the total length of the distinct states it contains."""
    total = 0
    found = states_in_grid(grid, states)
    for state in found:
        total += len(state)
    return total

In [5]:
def show_grid(grid):
    """Print a grid as the puzzle asks for it: rows reading across."""
    for row in grid:
        print(" ".join(row))


def show_score_report(grid, states=STATE_NAMES):
    """Print the grid, the states it contains, and the total score."""
    show_grid(grid)
    print()
    found = states_in_grid(grid, states)
    for state in sorted(found, key=len, reverse=True):
        print(f"{len(state):3d}  {state:<14}")
    print()
    print("total score:", score_grid(grid, states), "from", len(found), "states")

## Validate our scoring function

We will use the example grid.

In [6]:
EXAMPLE_GRID = [
    [BLANK, "I", "H"],
    ["D", "A", "O"],
    [BLANK, "W", BLANK],
]

show_score_report(EXAMPLE_GRID)

. I H
D A O
. W .

  5  IDAHO         
  4  OHIO          
  4  IOWA          

total score: 13 from 3 states


Our functions correctly find the 3 states and calculates the score.  

## Step 3: the model

OK, so the the model decisions are quite tricky here as we effectively need to implement our breadth first search in the CP SAT software (we can't just point the solver at our python function and expect it to do anything).  We'll add constraints for the letters (only one letter per square), and for the actual pathing.


### Letter variables

1. The letter constraint (only one letter per square) is straightforward.  We create 26 boolean variables for each cell (650 total), one for each letter. They will be true if the cell contains that letter, and false if it doesn't.  They will be keyed by letter and cell.

`cell_holds[row, column, letter]` return true if the specified cell holds letter, false otherwise.

Constraint: The sum of cell_holds for each cell must be 1.  

### Path variables

 This is a very tricky problem where we need to figure out how to express "this word can be walked". 

The frontier idea of BFS from `find_spelling_path` translates directly into variables: for each state, and each step of that state's spelling, and each square, create a boolean variable.  

`spelled_through[state][step][cell]` returns true if 

1. The cell contains the letter for that state at that step
2. `spelled_through[state][step-1][neighbor]` is true for any cell that is a neighbor. If step = 0, this constraint is removed.

Using this paradigm, we can then define `present[state]` as having some `spelled_through[state][len(state)][]` true.  

As an exercise, what this means that in a 3x3 grid, for the state 'OHIO', we have 36 path variables, one for each cell/letter combination.  
In our particular solution, `spelled_through['OHIO'][0][(0, 1)]` was true since that cell contains 'O', whereas `spelled_through['OHIO'][0][(1, 1)]` was false beacuse it contained A, and every other cell at step=0 is false except for (0,1).  `spelled_through['OHIO'][1][(0, 2)]` is also true, because it contains 'H' and it has a neighbor that was true for step=0.  Finally, `spelled_through['OHIO'][3][(1, 2)]` is true, meaning `present['OHIO']` is true, since at least 1 boolean for `OHIO` at max steps was true.  It's via the `present[STATE]` variables that the model actually picks up points to optimize.  

In [7]:
def build_model(size=GRID_SIZE, states=STATE_NAMES):
    """Build the Altered States model.

    Returns (model, cell_holds, state_is_present), where cell_holds[(row, column, letter)] is
    true when that square carries that letter, and state_is_present[state] is the Bool the
    objective pays for.
    """
    model = cp_model.CpModel()
    cells = all_cells(size)
    alphabet = ALPHABET

    cell_holds = {}  # Add our cell_holds variables
    for row, column in cells:
        for letter in alphabet:
            cell_holds[(row, column, letter)] = model.new_bool_var(
                f"cell_{row}{column}_holds_{letter}"
            )
        letters_here = [cell_holds[(row, column, letter)] for letter in alphabet]
        model.add_exactly_one(
            letters_here
        )  # this square must hold exactly one letter -> better optimized than summing them

    state_is_present = {}  # Add our state_is_present variables
    for state in states:
        # spelled_through[step][cell]: a legal walk spells state[:step + 1] and ends on cell.
        spelled_through = []
        for step, letter in enumerate(
            state
        ):  # returns (0, 'A'), (1, 'L'), etc. for 'ALABAMA'
            layer = {}
            for row, column in cells:
                reached = model.new_bool_var(
                    f"{state}_step{step}_at_{row}{column}"
                )  # Add a boolean for each state/step/cell triplet.

                # Whatever the walk did earlier, this square has to hold this step's letter.
                model.add_implication(
                    reached, cell_holds[(row, column, letter)]
                )  # Reached can only be true if the square holds the letter

                is_first_letter = step == 0
                if not is_first_letter:
                    # The previous letter has to sit on a square a King could have come from.
                    # king_neighbors excludes this square itself, which is what stops a walk
                    # from satisfying a repeated letter by standing still (the HAWAII rule).
                    arrivals = [
                        spelled_through[step - 1][neighbor]
                        for neighbor in king_neighbors(row, column, size)
                    ]
                    model.add_bool_or(arrivals).only_enforce_if(
                        reached
                    )  # Reached can only be true if at least one arrival is true

                layer[(row, column)] = reached
            spelled_through.append(layer)

        present = model.new_bool_var(f"present_{state}")
        final_step = spelled_through[-1]
        finishing_squares = [final_step[cell] for cell in cells]
        model.add_bool_or(finishing_squares).only_enforce_if(
            present
        )  # present can only be true for a state if at least one finishing square is true for that state
        state_is_present[state] = present

    return model, cell_holds, state_is_present


def add_score_objective(model, state_is_present):
    """Maximize the puzzle score, and return the score expression for later reuse."""
    score = sum(len(state) * present for (state, present) in state_is_present.items())
    model.maximize(score)
    return score

## How to speed it up

### Symmetry breaking
Since Rotating or reflecting the grid does not change its score, we enforce that the letter in top left is the "smallest" (removes the 4 rotations), and that the letter in position (0,1) is smaller than in position (1,0) (removes diagonal reflections).


### Some human obvious constraints. 

If a state is present, each of its letters must appear somewhere in the grid. The model doesn't know this explicitely so it's worth telling it.

In [8]:
def add_symmetry_breaking(model, cell_holds, size, alphabet):
    """Rule out rotated and reflected copies of a grid. Returns the per-cell letter-index vars."""
    index_of_letter = {}
    for index, letter in enumerate(alphabet):
        index_of_letter[letter] = index

    letter_index = {}
    for row, column in all_cells(size):
        index_here = model.new_int_var(
            0, len(alphabet) - 1, f"letter_index_{row}{column}"
        )
        for letter in alphabet:
            model.add(index_here == index_of_letter[letter]).only_enforce_if(
                cell_holds[(row, column, letter)]
            )
        letter_index[(row, column)] = index_here

    last = size - 1
    other_corners = [(0, last), (last, 0), (last, last)]
    for corner in other_corners:
        # Of the four corners, the top-left one holds the alphabetically smallest letter.
        # Some rotation of any grid satisfies this, so no score is lost.
        model.add(letter_index[(0, 0)] <= letter_index[corner])

    # The reflection across the main diagonal is the only symmetry that leaves the corner
    # ordering untouched, so it needs its own tie-break.
    model.add(letter_index[(0, 1)] <= letter_index[(1, 0)])

    return letter_index


def add_implied_letter_constraints(model, cell_holds, size, state_is_present):
    """Require every letter of a present state to appear somewhere in the grid."""
    for state, present in state_is_present.items():
        for letter in sorted(set(state)):
            squares_with_letter = [
                cell_holds[(row, column, letter)] for (row, column) in all_cells(size)
            ]
            model.add_bool_or(squares_with_letter).only_enforce_if(present)


def add_required_pairs(model, cell_holds, states, size=GRID_SIZE):
    """Require the grid to contain every adjacent letter pair these states need.

    NORTHCAROLINA needs an N beside an O, an O beside an R, and so on. The path variables in
    build_model already imply all of this, so it adds nothing to what the model *means* -- but
    the solver can check a pair in one step instead of deriving it through a chain of path
    variables, so it throws out hopeless partial grids far earlier.

    The pairs are worked out here rather than by calling pairs_of, which is defined further down
    the notebook alongside the overlap tooling. Keeping this self-contained means the cells can
    be run in any order.
    """
    needed_pairs = set()
    for state in states:
        for index in range(len(state) - 1):
            # Sorted, because a King's move works both ways: an A beside an R serves AR and RA.
            needed_pairs.add(tuple(sorted((state[index], state[index + 1]))))

    for first_letter, second_letter in sorted(needed_pairs):
        # One Bool per way the grid could realise this pair: some square holding the first
        # letter with a neighbour holding the second. Looping over every cell and every
        # neighbour covers both directions, which is why the sorted pair above loses nothing.
        realizations = []
        for row, column in all_cells(size):
            for neighbor in king_neighbors(row, column, size):
                neighbor_row, neighbor_column = neighbor
                realized_here = model.new_bool_var(
                    f"pair_{first_letter}{second_letter}"
                    f"_{row}{column}_{neighbor_row}{neighbor_column}"
                )
                # Claiming this pair is realised here means both squares really hold those
                # letters. As everywhere else, the implication only runs one way.
                model.add_implication(
                    realized_here, cell_holds[(row, column, first_letter)]
                )
                model.add_implication(
                    realized_here,
                    cell_holds[(neighbor_row, neighbor_column, second_letter)],
                )
                realizations.append(realized_here)

        # At least one of those squares pairs must actually be there.
        model.add_bool_or(realizations)

## Reading the solution

`grid_from_solver` turns the Bools back into a printable grid. 
`states_claimed_by_solver` reports what states the model thinks its grid can find.

In [9]:
def grid_from_solver(solver, cell_holds, size):
    """Return the solved grid as a list of rows of single-letter strings."""
    grid = []
    for row in range(size):
        grid.append([BLANK] * size)
    for (row, column, letter), holds_it in cell_holds.items():
        if solver.value(holds_it) == 1:
            grid[row][column] = letter
    return grid


def states_claimed_by_solver(solver, state_is_present):
    """Return the sorted states the model marked present."""
    claimed = []
    for state, present in state_is_present.items():
        if solver.value(present) == 1:
            claimed.append(state)
    return sorted(claimed)


def report_solution(
    solver, cell_holds, state_is_present, size=GRID_SIZE, states=STATE_NAMES
):
    """Print the solved grid and its independently re-checked score."""
    grid = grid_from_solver(solver, cell_holds, size)
    claimed = states_claimed_by_solver(solver, state_is_present)
    actually_present = sorted(states_in_grid(grid, states))

    show_score_report(grid, states)
    print()
    print("states the model claimed:", len(claimed))
    print("states the checker found:", len(actually_present))
    missed_by_model = sorted(set(actually_present) - set(claimed))
    if len(missed_by_model) > 0:
        # Expected and harmless: the objective had no reason to flag these, but they still score.
        print("present but unclaimed (free points):", missed_by_model)
    invented_by_model = sorted(set(claimed) - set(actually_present))
    # This one is never acceptable. It would mean the path encoding is wrong.
    assert len(invented_by_model) == 0, invented_by_model
    return grid

## Testing on the small grid

Before turning the solver loose on the real 5x5, I'll test my code by running it on the 3x3 grid and see what we get!

In [ ]:
small_model, small_cell_holds, small_present = build_model(3, STATE_NAMES)
add_score_objective(small_model, small_present)
small_solver = cp_model.CpSolver()
small_solver.parameters.max_time_in_seconds = 30
small_solver.parameters.num_workers = 8
small_solver.parameters.log_search_progress = True
small_status = small_solver.solve(small_model)
print(small_solver.status_name(small_status))

In [ ]:
best_grid = report_solution(small_solver, small_cell_holds, small_present, size=3)

It found an optimal solution! How exciting.  Nebraska, arkansas, and kansas are all share so many letters that makes sense.  Alabama is efficient with only 4 letters, as is alaska with the benefit of sharing a lot of the nebraska and arkansas letters.

## Solve the real grid and iterate!

I'll let the solver run for a few minutes to get a good baseline.  Then, I'll try to iterate by setting better bounds and using `add_hint` from the best grid found so far is helpful.

In [10]:
def add_grid_hint(model, cell_holds, grid, alphabet=ALPHABET):
    """Tell the solver to start its search from this grid rather than from scratch."""
    for row, column in all_cells(len(grid)):
        for letter in alphabet:
            if grid[row][column] == letter:
                model.add_hint(cell_holds[(row, column, letter)], 1)
            else:
                model.add_hint(cell_holds[(row, column, letter)], 0)


def show_grid_as_literal(grid):
    """Print the grid as Python source, ready to paste over BEST_GRID_SO_FAR."""
    print("BEST_GRID_SO_FAR = [")
    for row in grid:
        letters = ", ".join(f'"{letter}"' for letter in row)
        print(f"    [{letters}],")
    print("]")


def add_states(model, state_is_present, states):
    """Require every one of these states to be present, whatever it costs elsewhere."""
    for state in states:
        model.add(state_is_present[state] == 1)


def remove_states(model, state_is_present, states):
    """Stop the model claiming these states, so the objective never pays for them.

    This does not keep them out of the grid. The model only says present => a walk exists,
    never the reverse, so the grid may still contain them and score_grid will still find them.
    """
    for state in states:
        model.add(state_is_present[state] == 0)


def solve_once(
    seed_grid=None,
    fixed_squares=None,
    required_states=None,
    use_pair_constraints=True,
    minimum_score=None,
    seconds=120.0,
    random_seed=None,
):
    """Build a fresh model, solve it, and return (grid, score, status_name).

    seed_grid, if given, is where the search starts. minimum_score, if given, tells the solver
    not to bother with anything that bad, which prunes much harder than maximizing alone.
    fixed_squares, if given, is {(row, column): letter} for squares that must hold that letter.
    required_states, if given, are states the grid must contain, whatever that costs.
    use_pair_constraints adds the redundant adjacent-pair constraints for those states.
    Set it False to measure what they are worth: the best score must not move, only the time.

    Building a fresh model on every call is what makes this safe to re-run: nothing accumulates
    between calls, so these cells can be run in any order and as many times as I like.
    """
    model, cell_holds, state_is_present = build_model()
    score = add_score_objective(model, state_is_present)
    # Symmetry breaking assumes the grid is free to rotate and reflect. Pinning even one
    # square to a letter destroys that assumption, so the two can never be used together.
    if fixed_squares is None:
        add_symmetry_breaking(model, cell_holds, GRID_SIZE, ALPHABET)
    else:
        for (row, column), letter in fixed_squares.items():
            model.add(cell_holds[(row, column, letter)] == 1)
    add_implied_letter_constraints(model, cell_holds, GRID_SIZE, state_is_present)

    if required_states is not None:
        add_states(model, state_is_present, required_states)
        if use_pair_constraints:
            add_required_pairs(model, cell_holds, required_states, GRID_SIZE)
    if seed_grid is not None:
        add_grid_hint(model, cell_holds, seed_grid)
    if minimum_score is not None:
        model.add(score >= minimum_score)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = seconds
    solver.parameters.num_workers = 8
    solver.parameters.log_search_progress = True
    if random_seed is not None:
        solver.parameters.random_seed = random_seed
    status = solver.solve(model)

    # Asking for a minimum_score the grid cannot reach comes back with no solution at all.
    # Read nothing out of the solver in that case -- there is nothing in there to read.
    found_a_solution = status == cp_model.OPTIMAL or status == cp_model.FEASIBLE
    if not found_a_solution:
        return None, 0, solver.status_name(status)

    grid = grid_from_solver(solver, cell_holds, GRID_SIZE)

    # The model is allowed to understate a grid, but never to overstate one. A state it claims
    # that the plain-Python checker cannot find would mean the path encoding is broken.
    claimed = set(states_claimed_by_solver(solver, state_is_present))
    invented_by_model = sorted(claimed - states_in_grid(grid))
    assert len(invented_by_model) == 0, invented_by_model

    return grid, score_grid(grid), solver.status_name(status)

## The workhorse cell

I'll start my search by using this python cell to call 10 solvers to each run for 2 minutes, then take the best solution found.  Then, from that solution I'll try to start limiting the state space (literally lol) to see if that helps.

In [ ]:
BEST_SCORE_SO_FAR = 0
BEST_GRID_SO_FAR = None

for i in range(10):
    grid, found_score, status = solve_once(
        seed_grid=None,
        minimum_score=BEST_SCORE_SO_FAR + 1,
        seconds=120.0,
        random_seed=i,
    )
    print(status, found_score)

    found_something_better = grid is not None and found_score > BEST_SCORE_SO_FAR

    if found_something_better:
        BEST_GRID_SO_FAR = grid
        BEST_SCORE_SO_FAR = found_score
        show_score_report(BEST_GRID_SO_FAR)
        print()
        show_grid_as_literal(BEST_GRID_SO_FAR)
    else:
        print("no improvement on", BEST_SCORE_SO_FAR)

## Initial solution

The first pass found a score of 84

F L O R I
N C U N I
O S I A D
T G E W H
V I R O I

 12  WESTVIRGINIA  
  9  LOUISIANA     
  9  WISCONSIN     
  8  VIRGINIA      
  7  FLORIDA       
  7  INDIANA       
  7  GEORGIA       
  6  HAWAII        
  6  OREGON        
  5  IDAHO         
  4  OHIO          
  4  IOWA 

Our second round failed to improve, but our 3rd round broke through with 108!

K N S D N
R A K I A
D O L U H
C H T O S
N A R N M

 13  NORTHCAROLINA 
 13  SOUTHCAROLINA 
 11  NORTHDAKOTA   
 11  SOUTHDAKOTA   
  9  LOUISIANA     
  8  ARKANSAS      
  8  COLORADO      
  7  MONTANA       
  7  INDIANA       
  6  ALASKA        
  6  KANSAS        
  5  IDAHO         
  4  UTAH    


This run of 108 takes advantage of arkansas/kansas, dakota, and carolina arbitrage.  WE will now try to take advantage of those good, high scoring states.

## Taking advantage by restricting states

We see from our best solution that some states are efficient scorers because they overlap on NORTH and SOUTH or share lots of letters.

Lets impose NORTHCAROLINA, SOUTHCAROLINA,  NORTHDAKOTA, SOUTHDAKOTA, ARKANSAS, and KANSAS as those seem to be the most efficient scoring states.  Under that scaffolding, hopefully our model will be be more efficient and can find more high scoring solutions.  Going to run a single high depth search and take that as my best answer if it beats 108.  Otherwise, will just use the 108.

In [11]:
# Experiment: demand the six big overlapping states and let the solver maximize around them.
# Those six are worth 62 on their own, so anything it finds is 62 plus whatever it can fit in
# the cells they leave over. Delete this cell to drop the idea -- nothing else depends on it.
REQUIRED_STATES = [
    "NORTHCAROLINA",
    "SOUTHCAROLINA",
    "NORTHDAKOTA",
    "SOUTHDAKOTA",
    "ARKANSAS",
    "KANSAS",
]


grid, found_score, status = solve_once(
    required_states=REQUIRED_STATES,
    use_pair_constraints=True,
    seed_grid=None,
    seconds=600.0,
)
print(status, found_score)


Starting CP-SAT solver v9.15.6755
Parameters: max_time_in_seconds: 600 log_search_progress: true num_workers: 8

Initial optimization model '': (model_fingerprint: 0x4827e81593ab9d5f)
#Variables: 14'312 (#bools: 50 in objective) (14'281 primary variables)
  - 14'287 Booleans in [0,1]
  - 25 in [0,24]
#kBoolAnd: 16'924 (#enforced: 16'924) (#literals: 33'848)
#kBoolOr: 9'443 (#enforced: 9'420) (#literals: 64'690)
#kExactlyOne: 25 (#literals: 625)
#kLinear1: 631 (#enforced: 625)
#kLinear2: 4

Starting presolve at 0.00s
  1.82e-03s  0.00e+00d  [DetectDominanceRelations] 
  1.08e-02s  0.00e+00d  [PresolveToFixPoint] #num_loops=4 #num_dual_strengthening=2 
  5.40e-05s  0.00e+00d  [ExtractEncodingFromLinear] #potential_supersets=44 
  5.78e-04s  0.00e+00d  [DetectDuplicateColumns] 
  3.97e-04s  0.00e+00d  [DetectDuplicateConstraints] #duplicates=32 
[Symmetry] Graph for symmetry has 51'368 nodes and 133'937 arcs.
[Symmetry] Symmetry computation done. time: 0.008295 dtime: 0.0357454
[SAT pres

## Improved to 119

OK well that went well!  I found a solution of 119, which hit at 356 seconds.  That gives me confidence that the long run is doing work, and that maybe I should let it keep going!  I'm going to do one long run of 30 minutes to see if we can beat 119.  I'm going to add LOUISIANA AND INDIANA to the required states, taking advantage of the IANA

In [12]:
show_score_report(grid)
print()
show_grid_as_literal(grid)

C M N T C
Z O R A H
L I K T D
I A S O U
D N H I L

 13  SOUTHCAROLINA 
 13  NORTHCAROLINA 
 11  NORTHDAKOTA   
 11  SOUTHDAKOTA   
  9  LOUISIANA     
  8  ARKANSAS      
  8  COLORADO      
  7  ARIZONA       
  7  MONTANA       
  7  INDIANA       
  6  ALASKA        
  6  KANSAS        
  5  IDAHO         
  4  UTAH          
  4  OHIO          

total score: 119 from 15 states

BEST_GRID_SO_FAR = [
    ["C", "M", "N", "T", "C"],
    ["Z", "O", "R", "A", "H"],
    ["L", "I", "K", "T", "D"],
    ["I", "A", "S", "O", "U"],
    ["D", "N", "H", "I", "L"],
]


In [14]:
for i in range(3):
    grid, found_score, status = solve_once(
        seed_grid=None,
        required_states=REQUIRED_STATES,
        use_pair_constraints=True,
        seconds=480.0,
        random_seed=i,
    )
    print(status, found_score)
    show_score_report(grid)
    print()
    show_grid_as_literal(grid)


Starting CP-SAT solver v9.15.6755
Parameters: random_seed: 0 max_time_in_seconds: 480 log_search_progress: true num_workers: 8

Initial optimization model '': (model_fingerprint: 0x4827e81593ab9d5f)
#Variables: 14'312 (#bools: 50 in objective) (14'281 primary variables)
  - 14'287 Booleans in [0,1]
  - 25 in [0,24]
#kBoolAnd: 16'924 (#enforced: 16'924) (#literals: 33'848)
#kBoolOr: 9'443 (#enforced: 9'420) (#literals: 64'690)
#kExactlyOne: 25 (#literals: 625)
#kLinear1: 631 (#enforced: 625)
#kLinear2: 4

Starting presolve at 0.01s
  1.94e-03s  0.00e+00d  [DetectDominanceRelations] 
  2.41e-02s  0.00e+00d  [PresolveToFixPoint] #num_loops=4 #num_dual_strengthening=2 
  5.70e-05s  0.00e+00d  [ExtractEncodingFromLinear] #potential_supersets=44 
  1.03e-03s  0.00e+00d  [DetectDuplicateColumns] 
  4.70e-04s  0.00e+00d  [DetectDuplicateConstraints] #duplicates=32 
[Symmetry] Graph for symmetry has 51'368 nodes and 133'937 arcs.
[Symmetry] Symmetry computation done. time: 0.008363 dtime: 0.03

This search didn't improve, so gonna lock in the 119 as my final answer.